# Track B — 기존 신용정보 vs 대안변수 비교 및 씬파일러 자체 검증 (track_b_05 최종본)

담당: 서윤 | 선행 노트북: track_b_01~04

## 이 노트북을 처음 보는 사람을 위한 안내

이 노트북은 팀 기획서의 핵심 가설(H4, H5, H7, H8)과 강사님 피드백(④ 씬파일러 자체 검증)을 검증한 최종 기록이다. 여러 번의 시행착오(21개 임의분류, SCORE 사용, 오염된 필드 포함)를 거쳐 지금의 결과에 도달했고, 실패한 시도도 "왜 폐기했는지" 근거와 함께 그대로 남겨둔다.

## 최종 결론 요약

| 가설 | 결론 | 핵심 수치 |
|---|---|---|
| H4 (기존 대비 대안변수 개선) | ✅ 지지 | AUC 0.8076 → 0.8382 (+3.07%p) |
| H7 (대체 vs 보완) | ✅ 지지 | 기존정보 26개 중 25개 SHAP 순위 유지 |
| H8 (개별 변수 변별력) | ✅ 지지 | 대안변수 48개 전부 단독 AUC>0.5 |
| H5 (등급 고른 분산) | △ 부분지지 | 균등도 0.90(씬) vs 0.95(val), "고르게"는 과장이나 완전히 틀리진 않음 |
| 씬파일러 자체 검증 | ✅ 지지(간접) | 참고신호(n=240) 구분력, 69개 모델이 기존정보 단독보다 우수 |

## 기획서와의 정합성 — 어디가 같고 어디가 다른가

기획서 5.1(분석방법론)은 Track B의 정량 검증을 "**대안변수 모델과 기존 신용점수 등급의 예측력 비교**(AUROC, K-S)"라고 명시했다. 원래 계획은 실제 연체 라벨이 없어 **KCB Score를 등급화한 것을 목표변수(proxy)로 삼아 그 등급을 재현하는** 방식이었다(기획서 4.3).

**우리는 이 설계를 그대로 따르지 않았다.** track_b_01에서 KCB Score(원문 필드명 `PYE_SC0000000`, Track B) 자체가 연도 간 86.41% 변동한다는 걸 발견해, 신뢰할 수 없는 프록시라고 판단했다. 대신 실제 연체발생 라벨(`PYE_D1011000C`)을 찾아 정식 타겟으로 썼다. **이건 기획서를 어긴 것이 아니라, 기획서가 전제했던 "Track B엔 실제 연체 라벨이 없다"는 가정 자체가 우리 데이터에서는 성립하지 않아 방법론을 개선한 것이다** — 실제 라벨로 검증하는 것이 프록시(등급 재현)로 검증하는 것보다 방법론적으로 더 엄격하다.

이 노트북의 "기존 신용정보 vs 대안변수" 비교도 같은 맥락이다. KCB Score(완성된 점수 하나)를 기존 정보로 쓰는 대신, **그 점수를 구성하는 원재료에 해당하는 실제 신용거래 데이터(카드한도·대출잔액 등)**를 기존 정보로 재구성했다. 이 결정의 근거는 아래 섹션 4에서 출처와 함께 제시한다. 방향성(대안정보가 기존 정보를 보완해 예측력을 높인다는 것을 검증)은 기획서와 동일하며, 방법(무엇을 "기존 정보"로 쓸지)만 더 엄밀하게 재정의했다.

---
## 1. 환경설정 — 69개 대안변수 모델 로드 및 학습

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import gc

base_path = '/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터'
scan_path = f'{base_path}/통신카드CB_씬파일러.csv'

final_69_features = [
    'DAR', 'PYE_AL012G019', 'TOT_ASST', 'PYE_AL012G005', 'YOY_TOT_ASST_RTC',
    'R3M_ITRT_ENT_BROADCAST', 'AGE', 'OWN_HOUS_CNT', 'FAM_OWN_HOUS_CNT',
    'PYE_CAR_OWN', 'JB_TP', 'FST_CAR_ELPS', 'OWN_LIV_YN', 'FST_HOUS_BUY_ELPS',
    'FAM_OWN_LIV_YN', 'RET_SIL', 'HIGHEND_CD2', 'QOQ_TOT_ASST_RTC',
    'PYE_AL012G011', 'SHP_CNT', 'R3M_ITRT_SHOP_JIKGU', 'B1Y_MOB_OS',
    'HOME_ADM', 'R3M_ITRT_ENT_WEBTOON', 'R3M_ITRT_FIN_INSUR',
    'YOY_R3M_ITRT_COMM_VOIP_CS',
    'YOY_R3M_FOOD_AMT_RTC', 'R6M_MART_AMT', 'YOY_R3M_SSM_AMT_RTC',
    'YOY_R3M_MART_AMT_RTC', 'QOQ_R3M_MART_AMT_RTC',
    'YOY_R3M_HOUSEHOLD_AMT_RTC', 'YOY_R3M_CONV_AMT_RTC',
    'R6M_M_MART_AMT', 'YOY_R3M_M_CONV_AMT_RTC',
    'R6M_SSM_AMT', 'R6M_MED_AMT', 'R6M_HOUSEHOLD_AMT', 'R6M_OIL_AMT',
    'R6M_FOOD_AMT', 'R6M_CONV_AMT', 'R6M_BEAUTY_AMT', 'R6M_CUL_AMT',
    'R6M_M_CONV_AMT', 'QOQ_R3M_HOUSEHOLD_AMT_RTC', 'QOQ_R3M_E_COMM_AMT_RTC',
    'YOY_R3M_E_COMM_AMT_RTC_was_missing', 'YOY_R3M_STARBUCKS_AMT_RTC_was_missing',
    'CPYT_MART_AMT_RT', 'CPYT_CLOTHES_AMT_RT_was_missing', 'CPYT_HOUSEHOLD_AMT_RT',
    'CPYT_SSM_AMT_RT_was_missing', 'CPYT_M_CONV_AMT_RT', 'CPYT_OIL_AMT_RT_was_missing',
    'CPYT_SSM_AMT_RT', 'CPYT_E_COMM_AMT_RT', 'CPYT_STARBUCKS_AMT_RT_was_missing',
    'CPYT_MART_AMT_RT_was_missing', 'CPYT_STARBUCKS_AMT_RT',
    'grade_unavailable', 'PET_GD_woe', 'APP_GD_woe', 'GOLF_GD_woe', 'TRAVEL_GD_woe',
    'DAR_was_missing', 'FST_CAR_ELPS_was_missing', 'FST_HOUS_BUY_ELPS_was_missing',
    'CPYT_CONV_AMT_RT_was_missing', 'CPYT_FOOD_AMT_RT_was_missing'
]
final_69_features = list(dict.fromkeys(final_69_features))
print(f"69개 확인: {len(final_69_features)}개")

grade_source_cols = ['PET_GD', 'APP_GD', 'GOLF_GD', 'TRAVEL_GD']
thin_def_cols = ['PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004', 'PYE_C18233005', 'PYE_L10210000']
flag_derived = [c for c in final_69_features if c.endswith('_was_missing')]
raw_source_of_flags = [c.replace('_was_missing', '') for c in flag_derived]
direct_load = [c for c in final_69_features if not c.endswith('_was_missing')
               and c not in ['grade_unavailable','PET_GD_woe','APP_GD_woe','GOLF_GD_woe','TRAVEL_GD_woe']]

load_cols = list(dict.fromkeys(['CUST_ID'] + direct_load + raw_source_of_flags + grade_source_cols + thin_def_cols))
df = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv', usecols=load_cols)
gc.collect()

flag_source_df = df[raw_source_of_flags].isnull().astype(int)
flag_source_df.columns = [f'{c}_was_missing' for c in raw_source_of_flags]
df = pd.concat([df, flag_source_df], axis=1)
df[raw_source_of_flags] = df[raw_source_of_flags].fillna(0)
del flag_source_df
df[direct_load] = df[direct_load].fillna(0)
gc.collect()

thin_mask = (
    (df['PYE_C1M210000']==0) & (df['PYE_C18233003']==0) &
    (df['PYE_C18233004']==0) & (df['PYE_C18233005']==0) &
    (df['PYE_L10210000']==0)
)
df['segment'] = np.where(thin_mask, 'thin_filer', 'credit_holder')

df_y = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=['CUST_ID', 'PYE_D1011000C'])
df_y['TARGET'] = (df_y['PYE_D1011000C'] > 0).astype(int)
df = df.merge(df_y[['CUST_ID', 'TARGET']], on='CUST_ID', how='left')
del df_y
gc.collect()

credit_holders = df[df['segment']=='credit_holder'].copy()
thin_filers = df[df['segment']=='thin_filer'].copy()
del df
gc.collect()

credit_holders['grade_unavailable'] = (credit_holders['PET_GD'] == '*').astype(int)
thin_filers['grade_unavailable'] = (thin_filers['PET_GD'] == '*').astype(int)

def woe_encode_nonstar(df_, feature, target):
    data = df_[df_[feature] != '*'][[feature, target]].copy()
    grouped = data.groupby(feature, observed=True)[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']
    total_bad = grouped['bad'].sum()
    total_good = grouped['good'].sum()
    grouped['bad_rate'] = (grouped['bad'] + 0.5) / (total_bad + 0.5)
    grouped['good_rate'] = (grouped['good'] + 0.5) / (total_good + 0.5)
    grouped['woe'] = np.log(grouped['good_rate'] / grouped['bad_rate'])
    return grouped['woe'].to_dict()

for col in grade_source_cols:
    woe_map = woe_encode_nonstar(credit_holders, col, 'TARGET')
    credit_holders[f'{col}_woe'] = credit_holders[col].map(woe_map).fillna(0)
    thin_filers[f'{col}_woe'] = thin_filers[col].map(woe_map).fillna(0)

gc.collect()

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import xgboost as xgb

X = credit_holders[final_69_features]
y = credit_holders['TARGET']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model_full = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    eval_metric='auc', random_state=42
)
model_full.fit(X_train, y_train)
pred_proba = model_full.predict_proba(X_test)[:, 1]
print(f"[69개 대안변수 모델] Val AUC: {roc_auc_score(y_test, pred_proba):.4f}, Val PR-AUC: {average_precision_score(y_test, pred_proba):.4f}")

credit_holders['risk_score'] = model_full.predict_proba(credit_holders[final_69_features])[:, 1]
thin_filers['risk_score'] = model_full.predict_proba(thin_filers[final_69_features])[:, 1]
print(f"신용이력 보유군: {len(credit_holders):,}명, 씬파일러: {len(thin_filers):,}명")

Mounted at /content/drive
69개 확인: 69개
[69개 대안변수 모델] Val AUC: 0.7824, Val PR-AUC: 0.1820
신용이력 보유군: 285,890명, 씬파일러: 44,110명


---
## 2. "기존 신용정보"를 무엇으로 정의할지 — 근거를 먼저 확정한다

기획서가 원래 쓰려던 "기존 신용점수(KCB Score)"는 아래 이유로 이번 비교에서 완전히 배제한다.

**KCB Score(`PYE_SC0000000`)를 배제하는 근거 (출처 명시)**

1. **타겟(정답)으로 쓸 수 없음**: 연도 간(2021→2022) 값 변동 86.41% — track_b_01에서 직접 계산해 확인.
2. **씬파일러에게는 존재하지 않는 값**: 씬파일러 6.95%는 0점, 나머지도 신뢰 불가 — track_b_01 코드북 교차검증.
3. **정확한 출처 확인**: 코드북 Sheet09(개인CB정보, Track A)의 필드 `SCORE`에는 "K-Score2.0 적용배제대상 고객은 0점으로 제공, 그 외 고객은 평점 제공"이라는 설명이 명시되어 있다(엑셀 파일 `093-117_금융 합성데이터_데이터구성상세.xlsx`, Sheet명 "9.개인CB 정보", 필드 `SCORE` 행). **다만 이 설명은 Track A의 필드에 달린 것이고, Track B에서 쓴 `PYE_SC0000000`(Sheet11, "11.통신카드CB 결합정보")에는 이런 설명이 없다** — 같은 K-Score 체계일 가능성이 높지만 공식 문서로 직접 확인되지는 않았다는 한계가 있다. 이 불확실성도 SCORE를 최종 비교에서 제외하는 이유 중 하나다.

**대신 무엇을 "기존 신용정보"로 쓸 것인가 — 근거: 여신심사 5C 원칙**

한국 금융권 여신심사역 자격시험 교재(신협, 출처: quizgecko.com "신협 여신심사역 시험 대비")에 따르면, 여신심사의 국제표준 원칙인 **5C**는 다음과 같다:

> "Capacity(상환능력)는 차주가 대출금을 원리금과 함께 상환할 수 있는 능력을 의미하며, **주로 소득, 현금흐름, 부채 수준 등을 분석**하여 평가합니다."
> "Capital(자본)은 차주의 **자기 자본 규모**를 의미하며, 이는 차주의 재무 안정성과 위험 감수 능력을 보여줍니다."

또한 카카오뱅크 공식 블로그(brunch.co.kr/@dambee/74, "KCB NICE 신용점수 등급 다르게 나오는 이유는?")에 따르면, 실제 KCB/NICE 개인신용평점은 **상환이력·부채수준(24.5%)·신용거래기간(12.3%)·신용거래형태**로 구성되며, "나이·직업·학력 등의 민감정보는 신용평가에 반영되지 않는다"(NICE평가정보 공식 페이지, niceinfo.co.kr/creditrating/cb_score_1_4_1.nice)고 명시되어 있다.

**따라서**: 부채수준·신용거래형태·신용거래기간에 해당하는 실제 금융거래 데이터(카드한도, 카드이용률, 대출잔액, 대출건수 등)를 "기존 신용정보"로 채택한다. 이 필드들은 track_b_02에서 씬파일러 정의와의 순환논리 우려로 예측변수 후보에서 제외됐던 "신용노출" 필드들이다 — 신용이력 보유군(모두 카드·대출 보유자)만 대상으로 하는 이번 분석에서는 그 순환논리 우려가 적용되지 않는다(씬파일러 vs 신용이력보유군을 나누는 게 아니라, 신용이력보유군 내에서 연체를 예측하는 것이므로).

In [2]:
traditional_credit_features = [
    'PYE_C1L120161',  # 최초카드개설일로부터경과일수 - 신용거래기간
    'PYE_C1L120012', 'PYE_C1L120196',  # 카드총한도금액(현재/1년전) - 부채수준
    'HOUS_LN_BAL', 'CRDT_LN_BAL', 'SHP_LN_BAL',  # 대출잔액 - 부채수준
    'PYE_L1021003P', 'PYE_L1021006P', 'PYE_L102100CP',  # 기간별 대출건수 - 신용거래형태
    'PYE_L90210100', 'PYE_L90210200', 'PYE_L90210300',
    'PYE_L10210800', 'PYE_L10210B00', 'PYE_L10216000',
    'PYE_L90216100', 'PYE_L90216300', 'PYE_L10216800', 'PYE_L10216B00',
    'PYE_L10231000',  # 총약정금액
    'QOQ_HOUS_LN_BAL_RTC', 'QOQ_CRDT_LN_BAL_RTC', 'QOQ_SHP_LN_BAL_RTC',
    'YOY_HOUS_LN_BAL_RTC', 'YOY_CRDT_LN_BAL_RTC', 'YOY_SHP_LN_BAL_RTC',
]
# 의도적으로 제외: CD_USE_AMT(카드소비총액, 순환논리 확인됨), PYE_L1021A000(연체대환대출건수, 필드명에 연체 포함되어 타겟과 근접),
#                씬파일러 정의 5개 필드(PYE_C1M210000/PYE_C18233003~5/PYE_L10210000, 순환논리 방지)
print(f"기존 신용정보(1차): {len(traditional_credit_features)}개")

기존 신용정보(1차): 26개


---
## 3. (중요, 재현 필수) 값 오염 검증 — PYE_C1L120237, PYE_C1L120250 제외

**첫 시도(28개)에서 발견된 문제**: `PYE_C1L120237`(카드이용잔액 유효한도소진율), `PYE_C1L120250`(카드론 이용잔액 유효한도소진율)을 포함시켰더니, **카드가 전혀 없는 씬파일러(카드건수=0) 44,110명 중 74%가 정확히 "153" 또는 "156"이라는 동일한 값을 가지고 있었다.** 카드가 없는 사람에게 "카드 이용률"이 정상적인 비율값으로 나올 수 없으므로, 이 숫자들은 실제 비율이 아니라 코드북에 설명되지 않은 **특수 코드값(추정: 정보없음/계산불가 등을 나타내는 센티널 값)**으로 판단된다.

**검증 방법**: 신용이력 보유군(카드 있는 사람)에서도 같은 값(153/156)에 사람이 몰리는지 확인 — 카드 있는 사람 280,529명 중 24,979명(153)과 18,207명(156)에 몰려 있고, 그 외 값은 전부 소수점 단위로 3명 이하씩만 분포. **정상적인 연속형 비율값이라면 이렇게 두 값에만 수만 명이 몰릴 수 없다.** 코드북(Sheet11)에는 이 필드에 대한 코드 설명이 없어(코드여부: N), 정확한 의미는 확인 불가 — 신뢰할 수 없는 필드로 판단해 제외한다.

**최종 26개**: 위 28개에서 이 2개 필드(및 파생 `_was_missing` 플래그)를 제외.

In [3]:
traditional_credit_features_clean = [f for f in traditional_credit_features
                                       if f not in ['PYE_C1L120237', 'PYE_C1L120250']]
print(f"최종 기존 신용정보: {len(traditional_credit_features_clean)}개")

df_traditional = pd.read_csv(f'{scan_path}/202103_통신카드CB결합.csv',
                               usecols=['CUST_ID'] + traditional_credit_features_clean)

missing_trad = df_traditional[traditional_credit_features_clean].isnull().sum()
missing_trad = missing_trad[missing_trad > 0]
print(f"결측 있는 컬럼: {len(missing_trad)}개")
print(missing_trad)

for col in missing_trad.index:
    df_traditional[f'{col}_was_missing'] = df_traditional[col].isnull().astype(int)
    df_traditional[col] = df_traditional[col].fillna(0)

traditional_flag_cols = [c for c in df_traditional.columns if c.endswith('_was_missing')]
traditional_all_clean = traditional_credit_features_clean + traditional_flag_cols

# 순환논리 최종 확인 - 씬파일러 정의 5개 필드와 겹치는 게 없는지
thin_def_5 = ['PYE_C1M210000', 'PYE_C18233003', 'PYE_C18233004', 'PYE_C18233005', 'PYE_L10210000']
overlap = set(traditional_all_clean) & set(thin_def_5)
print(f"\n씬파일러 정의 필드와 겹치는 것: {overlap if overlap else '없음 (순환논리 없음, 안전)'}")

credit_holders_trad_clean = credit_holders.merge(df_traditional, on='CUST_ID', how='left')
print(f"\ncredit_holders_trad_clean 결측 총합: {credit_holders_trad_clean[traditional_all_clean].isnull().sum().sum()}")

최종 기존 신용정보: 26개
결측 있는 컬럼: 8개
PYE_C1L120012          203523
PYE_C1L120196          203523
QOQ_HOUS_LN_BAL_RTC    281681
QOQ_CRDT_LN_BAL_RTC    223741
QOQ_SHP_LN_BAL_RTC     307694
YOY_HOUS_LN_BAL_RTC    282177
YOY_CRDT_LN_BAL_RTC    227675
YOY_SHP_LN_BAL_RTC     311527
dtype: int64

씬파일러 정의 필드와 겹치는 것: 없음 (순환논리 없음, 안전)

credit_holders_trad_clean 결측 총합: 0


---
## 4. H4 검증 — 기존 신용정보만 vs 기존+대안변수 (신용이력 보유군)

이게 기획서 5.1이 요구한 "대안변수 모델과 기존 정보의 예측력 비교"에 해당하는 검증이다. KCB Score 대신, 섹션 2~3에서 근거를 확인한 26개 실제 신용거래 데이터를 "기존 정보"로 사용한다.

In [4]:
def train_and_eval_trad(df_source, feature_list, label):
    X_ = df_source[feature_list]
    y_ = df_source['TARGET']
    X_tr, X_te, y_tr, y_te = train_test_split(X_, y_, test_size=0.2, stratify=y_, random_state=42)
    m = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                            scale_pos_weight=(y_tr==0).sum()/(y_tr==1).sum(),
                            eval_metric='auc', random_state=42)
    m.fit(X_tr, y_tr)
    proba = m.predict_proba(X_te)[:, 1]
    auc_ = roc_auc_score(y_te, proba)
    pr_ = average_precision_score(y_te, proba)
    print(f"[{label}] 변수 {len(feature_list)}개 -> AUC {auc_:.4f}, PR-AUC {pr_:.4f}")
    return m, auc_, pr_

model_traditional, auc_trad, pr_trad = train_and_eval_trad(
    credit_holders_trad_clean, traditional_all_clean, "기존 신용정보만(26개)"
)

combined_features = list(dict.fromkeys(traditional_all_clean + final_69_features))
model_combined, auc_combined, pr_combined = train_and_eval_trad(
    credit_holders_trad_clean, combined_features, "기존+대안변수 69개"
)

print(f"\n=== H4 최종 결과 ===")
print(f"[기존 신용정보만]     AUC {auc_trad:.4f}, PR-AUC {pr_trad:.4f}")
print(f"[기존+대안변수]       AUC {auc_combined:.4f}, PR-AUC {pr_combined:.4f}")
print(f"대안변수 추가 효과: AUC {auc_combined-auc_trad:+.4f}, PR-AUC {pr_combined-pr_trad:+.4f}")
# 기록치: AUC 0.8076 -> 0.8382 (+3.07%p), PR-AUC 0.2418 -> 0.2782 (+3.64%p)

[기존 신용정보만(26개)] 변수 34개 -> AUC 0.8076, PR-AUC 0.2418
[기존+대안변수 69개] 변수 103개 -> AUC 0.8382, PR-AUC 0.2782

=== H4 최종 결과 ===
[기존 신용정보만]     AUC 0.8076, PR-AUC 0.2418
[기존+대안변수]       AUC 0.8382, PR-AUC 0.2782
대안변수 추가 효과: AUC +0.0307, PR-AUC +0.0364


**결과 해석**: 대안변수 추가로 AUC +3.07%p, PR-AUC +3.64%p 개선. 기획서가 인용한 선행연구(NICE, 2023: 신용점수만 67.8% → 대안정보 추가 76.8%, +9.0%p)와 같은 방향의 패턴이며, 우리 baseline(0.8076)이 이미 상당히 높은 지점에서 시작하는 걸 감안하면 합리적인 개선폭이다.

---
## 4-1. 비교용 CSV 저장 — 모델링 담당자 전달용

**중요**: 이 26개(기존 신용정보) 모델은 최종 서비스 모델이 아니라, H4/H7 비교 검증 전용이다. 지금까지의 검증(섹션 2~4)은 고정 하이퍼파라미터·1회 train/test 분할로 진행한 **1차 방향성 확인**일 뿐이며, 튜닝·교차검증·알고리즘 비교(로지스틱/랜덤포레스트 병행, 기획서 5.3)를 거친 정식 수치는 아니다. 모델링 담당자가 이 CSV를 가지고 정식으로 재검증해서, H4(+3.07%p 등)의 최종 확정 수치를 내야 한다.

**씬파일러용은 별도로 만들지 않는다** — 이 26개는 씬파일러에게 대부분 0(카드·대출 없음)으로 구조적으로 무의미한 값이라(섹션 8 확인), 비교 검증은 신용이력 보유군만으로 충분하다.

In [5]:
# ===== 비교용 CSV — 기존 신용정보 26개 (신용이력 보유군만, H4/H7 비교 전용) =====

export_cols_traditional = ['CUST_ID'] + traditional_all_clean + ['TARGET']
df_traditional_export = credit_holders_trad_clean[export_cols_traditional]

print(f"결측 확인: {df_traditional_export[traditional_all_clean].isnull().sum().sum()}")

df_traditional_export.to_csv(
    f'{base_path}/변수스캔결과/track_b_traditional_credit_features_for_H4comparison.csv',
    index=False
)
print(f"저장 완료: {df_traditional_export.shape}")
print("\n※ 이 파일은 최종 모델용이 아니라 H4/H7 비교 검증 전용입니다.")
print("  최종 모델용 파일은 track_b_features_train_v2.csv / apply_v2.csv (69개 변수)입니다.")

결측 확인: 0
저장 완료: (285890, 36)

※ 이 파일은 최종 모델용이 아니라 H4/H7 비교 검증 전용입니다.
  최종 모델용 파일은 track_b_features_train_v2.csv / apply_v2.csv (69개 변수)입니다.


---
## 5. H7 검증 — 대체 vs 보완 (기존 정보의 SHAP 순위가 대안변수 추가 후 유지되는가)

In [6]:
import shap

explainer_trad = shap.TreeExplainer(model_traditional)
X_trad_test = credit_holders_trad_clean[traditional_all_clean].sample(5000, random_state=42)
shap_trad = explainer_trad.shap_values(X_trad_test)
imp_trad = pd.Series(np.abs(shap_trad).mean(axis=0), index=traditional_all_clean).sort_values(ascending=False)

explainer_combined = shap.TreeExplainer(model_combined)
X_combined_test = credit_holders_trad_clean[combined_features].loc[X_trad_test.index]
shap_combined = explainer_combined.shap_values(X_combined_test)
imp_combined_all = pd.Series(np.abs(shap_combined).mean(axis=0), index=combined_features)
imp_combined_trad_only = imp_combined_all[traditional_all_clean].sort_values(ascending=False)

compare_h7 = pd.DataFrame({
    '기존정보_단독_SHAP': imp_trad,
    '대안변수_추가후_SHAP': imp_combined_trad_only
})
compare_h7['순위_단독'] = compare_h7['기존정보_단독_SHAP'].rank(ascending=False)
compare_h7['순위_추가후'] = compare_h7['대안변수_추가후_SHAP'].rank(ascending=False)
compare_h7['순위변화'] = compare_h7['순위_단독'] - compare_h7['순위_추가후']
print(compare_h7.round(4).sort_values('순위_단독'))

big_moves = compare_h7[compare_h7['순위변화'].abs() >= 5]
print(f"\n순위가 크게 바뀐 변수(±5 이상): {len(big_moves)}개")
print(big_moves[['순위_단독','순위_추가후','순위변화']])
# 기록치: 1위 PYE_C1L120012(카드총한도) 그대로 유지, 34개 중 33개가 ±5 이내 유지, 예외 1개(하위권, SHAP값 매우 작음)

                                 기존정보_단독_SHAP  대안변수_추가후_SHAP  순위_단독  순위_추가후  \
PYE_C1L120012                          0.9221         0.5701    1.0     1.0   
PYE_C1L120196                          0.4063         0.1245    2.0     6.0   
CRDT_LN_BAL                            0.2684         0.1935    3.0     4.0   
PYE_C1L120161                          0.2234         0.0868    4.0     8.0   
PYE_L102100CP                          0.2215         0.2293    5.0     2.0   
YOY_CRDT_LN_BAL_RTC_was_missing        0.2091         0.2107    6.0     3.0   
PYE_C1L120196_was_missing              0.1639         0.1407    7.0     5.0   
PYE_L10210800                          0.1217         0.1048    8.0     7.0   
PYE_L10231000                          0.1134         0.0549    9.0    13.0   
PYE_C1L120012_was_missing              0.1113         0.0664   10.0    11.0   
PYE_L1021003P                          0.1065         0.0510   11.0    14.0   
PYE_L90216100                          0.0982       

**결과 해석**: 1위(카드총한도금액)를 포함해 대부분의 기존 신용정보 변수가 대안변수 추가 후에도 순위를 유지했다(34개 중 33개가 ±5 이내). 유일한 예외는 SHAP 기여도가 매우 작은 하위권 변수 하나뿐이다. **대안정보는 기존 신용정보를 대체하지 않고 보완하는 성격이 강하다는 H7이 지지된다.**

---
## 6. H8 검증 — 개별 대안변수 단위 변별력 (신용이력 보유군)

48개 대안변수 각각이 단독으로도 무작위(AUC 0.5)보다 유의미한 예측력을 갖는지 확인한다. baseline 개념 없이도 성립하는, 가장 근거가 튼튼한 검증이다.

In [7]:
baseline_features_for_split = [
    'DAR', 'TOT_ASST', 'YOY_TOT_ASST_RTC', 'QOQ_TOT_ASST_RTC', 'DAR_was_missing',
    'AGE', 'JB_TP', 'OWN_HOUS_CNT', 'FAM_OWN_HOUS_CNT', 'OWN_LIV_YN', 'FAM_OWN_LIV_YN',
    'PYE_CAR_OWN', 'FST_CAR_ELPS', 'FST_HOUS_BUY_ELPS',
    'FST_CAR_ELPS_was_missing', 'FST_HOUS_BUY_ELPS_was_missing',
    'HIGHEND_CD2', 'SHP_CNT', 'RET_SIL', 'HOME_ADM', 'B1Y_MOB_OS'
]
# 주의: 이 21개는 H4/H7의 "기존 정보"가 아니라, H8에서 "행동 기반 대안변수만" 추출하기 위한 참고용 리스트일 뿐이다.
# (섹션 2~5에서 이미 밝혔듯, 이 21개를 "전통적 신용평가 요소"라고 부르는 것은 근거가 없어 폐기했다.
#  여기서는 그냥 "69개 중 자산/인구통계 성격이 아닌 나머지"를 골라내는 용도로만 사용한다.)
alternative_only = [f for f in final_69_features if f not in baseline_features_for_split]
print(f"H8 검증 대상(행동 기반 대안변수): {len(alternative_only)}개")

individual_auc = []
for feat in alternative_only:
    X_single = credit_holders[[feat]]
    y_single = credit_holders['TARGET']
    X_tr, X_te, y_tr, y_te = train_test_split(X_single, y_single, test_size=0.2, stratify=y_single, random_state=42)
    m = xgb.XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                            scale_pos_weight=(y_tr==0).sum()/(y_tr==1).sum(),
                            eval_metric='auc', random_state=42)
    m.fit(X_tr, y_tr)
    proba = m.predict_proba(X_te)[:, 1]
    auc_single = roc_auc_score(y_te, proba)
    individual_auc.append({'변수명': feat, '단일변수_AUC': auc_single})

individual_auc_df = pd.DataFrame(individual_auc).sort_values('단일변수_AUC', ascending=False)
print(individual_auc_df.head(10).to_string(index=False))
print(f"\n단일변수 AUC 0.5 초과 개수: {(individual_auc_df['단일변수_AUC']>0.5).sum()} / {len(individual_auc_df)}")
# 기록치: 48/48 전부 0.5 초과. 1위 PYE_AL012G005(자택주소 이력, 0.592)

H8 검증 대상(행동 기반 대안변수): 48개
                         변수명  단일변수_AUC
               PYE_AL012G005  0.592295
               PYE_AL012G019  0.579893
      R3M_ITRT_ENT_BROADCAST  0.567493
                  PET_GD_woe  0.548834
                  APP_GD_woe  0.542637
CPYT_CONV_AMT_RT_was_missing  0.542461
CPYT_FOOD_AMT_RT_was_missing  0.541091
        YOY_R3M_FOOD_AMT_RTC  0.540528
 CPYT_SSM_AMT_RT_was_missing  0.540171
                 GOLF_GD_woe  0.538939

단일변수 AUC 0.5 초과 개수: 48 / 48


---
## 7. H5 검증 — 씬파일러 등급이 "고르게 분산"되는가

기획서 H5: "씬파일러는 기존 방식에서는 중간 등급에 몰리지만, 대안변수 기반 모델에서는 등급이 고르게 분산된다." ChiMerge/IV로 만든 9등급(track_b_05 이전 섹션, 여기선 결과만 인용)에서:
```
씬파일러 등급 분포: 1등급 5.82%, 2등급 23.76%, 3등급 18.35%, 4등급 17.91%, 5등급 15.95%, 6등급 5.67%, 7등급 4.45%, 8등급 3.18%, 9등급 4.92%
```
**균등도(엔트로피 기준, 1이 완전균등)**: 씬파일러 0.9001, 신용이력 보유군(val) 0.9537. 등급 3~5(중간) 합산 씬파일러 52.21% vs val 35.91%(약 1.45배).

**해석**: "완전히 고르게 분산된다"는 H5의 표현은 과장이다 — 균등도 자체는 0.90으로 1에 가까워 극적으로 나쁘지 않지만, val 대비 중간 등급에 상대적으로 더 몰리는 경향은 분명 존재한다(1.45배). **H5는 부분적으로만 지지된다**: 씬파일러가 완전히 평가불가 상태에서 벗어나 실제 등급 분산을 갖게 된 것은 사실이나(섹션 8 참고), "고르게" 분산된다는 표현은 정확하지 않다.

---
## 8. 씬파일러 자체 검증 (강사님 피드백 ④) — 참고신호(n=240) 기준 최종 확인

**왜 필요한가**: 씬파일러는 정답(TARGET)이 없다. track_b_01에서 5가지 방법으로 정식 타겟 구성을 시도했으나 불가능했다(신규대출 발생 0명 등). 유일하게 관측 가능한 건 `PYE_D1011000C > 0`(연체 유사 흔적)인 240명 — apply.csv엔 정식 TARGET으로 넣지 않은, 이 검증에서만 참고용으로 쓰는 신호다.

**240명이 정확히 누구인가**: 2020년 말(202103) 씬파일러였던 44,110명 중 2021년(202203)에 이 흔적이 생긴 사람들. 그중 176명(73%)은 2021년에도 씬파일러를 유지, 64명(27%)은 카드·대출을 신규 개설하며 씬파일러를 벗어났다(둘 다 위험점수 계산은 2020년말 정보로만 함, 시점 누출 없음).

In [8]:
cols_ref = ['CUST_ID', 'PYE_D1011000C']
df_ref = pd.read_csv(f'{scan_path}/202203_통신카드CB결합.csv', usecols=cols_ref)
df_ref['ref_y'] = (df_ref['PYE_D1011000C'] > 0).astype(int)
thin_filers_ref = thin_filers.merge(df_ref[['CUST_ID', 'ref_y']], on='CUST_ID', how='left')
print(f"참고용 y=1: {thin_filers_ref['ref_y'].sum()}명 / y=0: {(thin_filers_ref['ref_y']==0).sum()}명")

# 69개(대안변수 포함) 모델 기준 구분력
group_y1_full = thin_filers_ref[thin_filers_ref['ref_y']==1]['risk_score']
group_y0_full = thin_filers_ref[thin_filers_ref['ref_y']==0]['risk_score']
gap_69 = group_y1_full.mean() - group_y0_full.mean()

from scipy.stats import mannwhitneyu
_, pvalue = mannwhitneyu(group_y1_full, group_y0_full, alternative='greater')
print(f"\n[69개 모델] y=1 평균 {group_y1_full.mean():.4f} / y=0 평균 {group_y0_full.mean():.4f} / 구분력 {gap_69:.4f} / p={pvalue:.6f}")

# 기존 신용정보(26개) 모델을 씬파일러에 적용 - 비교 기준
thin_filers_trad = thin_filers.merge(df_traditional, on='CUST_ID', how='left')
thin_filers_trad['risk_score_trad'] = model_traditional.predict_proba(thin_filers_trad[traditional_all_clean])[:, 1]
thin_filers_trad_ref = thin_filers_trad.merge(df_ref[['CUST_ID', 'ref_y']], on='CUST_ID', how='left')

group_y1_trad = thin_filers_trad_ref[thin_filers_trad_ref['ref_y']==1]['risk_score_trad']
group_y0_trad = thin_filers_trad_ref[thin_filers_trad_ref['ref_y']==0]['risk_score_trad']
gap_trad = group_y1_trad.mean() - group_y0_trad.mean()
print(f"[기존정보(26개) 모델] y=1 평균 {group_y1_trad.mean():.4f} / y=0 평균 {group_y0_trad.mean():.4f} / 구분력 {gap_trad:.4f}")

print(f"\n씬파일러 risk_score 표준편차 - 69개 모델: {thin_filers['risk_score'].std():.4f}")
print(f"씬파일러 risk_score_trad 표준편차 - 기존정보(26개): {thin_filers_trad['risk_score_trad'].std():.4f}")
# 기록치(28개 버전 기준, 26개로도 방향 동일 예상): 69개 구분력 0.1556 > 기존정보 단독 구분력 0.1203, 표준편차도 69개가 2배 이상 큼

참고용 y=1: 240명 / y=0: 43870명

[69개 모델] y=1 평균 0.5278 / y=0 평균 0.3723 / 구분력 0.1556 / p=0.000000
[기존정보(26개) 모델] y=1 평균 0.4393 / y=0 평균 0.3517 / 구분력 0.0876

씬파일러 risk_score 표준편차 - 69개 모델: 0.1559
씬파일러 risk_score_trad 표준편차 - 기존정보(26개): 0.0461


**결과 해석**: 씬파일러는 정의상 기존 신용정보(카드한도·대출잔액 등)가 대부분 0인 구조적 결측 상태다. 이 정보만으로 만든 모델은 씬파일러 내 위험점수 변별력이 낮다(표준편차가 69개 모델의 절반 이하 수준). 반면 69개(대안변수 포함) 모델은 참고신호(n=240) 기준으로 뚜렷하게 더 높은 구분력을 보였다(p<0.001). **이는 "씬파일러에게는 기존 신용정보가 사실상 무의미하고, 대안변수가 유일하게 작동하는 평가 수단"이라는 이 프로젝트의 핵심 주장을 뒷받침한다.**

**한계**: n=240은 극소표본, 정식 라벨 아님(apply.csv에 미포함), 상관관계이지 인과관계 아님, 선택편향 가능성(왜 이 240명에게만 흔적이 남았는지 불명).

---
## 9. 폐기된 시도들 — 기록용 (다른 조원 설명용)

**① SCORE(KCB Score)를 기준으로 쓰려던 시도**: 두 가지 이유로 최종 폐기.
- 타겟(정답)으로는 처음부터 못 씀 — 연도간 86.41% 변동(track_b_01).
- 비교 기준(baseline)으로도 못 씀 — 씬파일러에게는 존재하지 않는 값(0점 또는 신뢰불가)이라, 이 프로젝트가 답해야 할 대상(씬파일러)에 대해 아무 정보도 주지 못함.
- (참고로 SCORE를 baseline으로 잠깐 써봤을 때 예측력은 낮고 안정성은 높다는 트레이드오프도 확인했으나, 이건 폐기 이유가 아니라 부가 관찰이었다.)

**② 21개(재무·인구통계)를 "전통적 신용평가 요소"라고 부르며 baseline으로 쓴 시도**: 실제 KCB 신용평가는 상환이력·부채수준·신용거래기간·신용거래형태로 구성되며 **나이·직업·학력 등은 반영되지 않는다**(NICE평가정보 공식페이지 확인). 즉 나이·직업·주택보유를 "전통적 신용평가 요소"라 부른 것은 근거 없는 주장이었다. 업계 정의(퍼블릭뉴스·뉴스토마토·헤럴드경제 등 여러 기사 확인)로도 "전통적 신용평가=실제 대출/카드/연체 거래기록"이고, 소득·자산·인구통계는 이 구도의 "전통"에도 "대안"에도 명확히 속하지 않는 애매한 카테고리였다. 폐기.

**③ 신용노출 필드 28개(오염 버전)**: `PYE_C1L120237`, `PYE_C1L120250` 두 필드에 코드북에 설명되지 않은 특수 코드값(153/156, 2358/2458)이 다수 섞여 있어, 제외하고 26개로 재계산(섹션 3 참고).

---
## 10. 모델링 담당자(김태헌) 전달사항

- 위 모든 분석은 전처리 단계의 변수 정당화·가설 검증용 탐색이며, 정식 모델(알고리즘 비교, 하이퍼파라미터 튜닝, 교차검증, WOE/IV 스코어카드)은 별도 구축 필요.

**전달 파일 2종 (용도가 다름, 헷갈리지 말 것)**
1. `track_b_features_train_v2.csv` / `apply_v2.csv` (69개 변수) → **실제 최종 서비스 모델용**
2. `track_b_traditional_credit_features_for_H4comparison.csv` (26개 변수, 신용이력 보유군만) → **H4/H7 비교 검증 전용, 최종 모델 아님.** 이 노트북 섹션 4/5에서 낸 결과(AUC 0.8076→0.8382, SHAP 순위 유지)는 고정 하이퍼파라미터·1회 분할로 낸 1차 방향성 확인일 뿐, 정식 수치가 아니다. **모델링 담당자가 튜닝·교차검증·알고리즘 비교(로지스틱/랜덤포레스트 병행)를 거쳐 H4/H7의 최종 확정 수치를 다시 내야 한다.**

- 섹션 4의 "기존 신용정보 26개" 목록 자체(어떤 필드를 왜 골랐는지, 오염 필드 제외 근거)는 섹션 2~3에 근거와 함께 정리되어 있으니 그대로 재사용 가능 — 필드를 다시 찾을 필요는 없음.
- SCORE, 21개 임의분류는 모두 폐기됐으므로 참고하지 말 것(섹션 9).